# HFSS direct central-difference local gradients

This standalone notebook evaluates the physical coordinate axes $e_i$. For $d$ active parameters, each step scale requires $2d$ HFSS calls. The center evaluation is shared by all scales, so three scales require $6d+1$ calls (79 calls for 13 active parameters with `repeats=1`). `gradient[i]` is the physical derivative for `PARAM_NAMES[i]`, for example $\partial S11/\partial h1$.

`sigma_vec` describes manufacturing variation and weights susceptibility. The finite-difference step is a separate numerical choice. All configured optimization boundaries are intentionally ignored: every active parameter must have two distinct rounded perturbation points, $x_i-h_i$ and $x_i+h_i$. The design raises an error before HFSS starts if the requested step cannot produce that symmetric pair at the configured input precision. Forward and backward difference formulas are not used.

The default scales are `0.25`, `0.5`, and `1.0` times sigma. Run the cells from top to bottom. Editing `x_center` in **User settings** is intentionally direct. After the complete perturbation design is validated, saved, and displayed, the following cell starts the HFSS evaluations.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone

%load_ext autoreload
%autoreload 2


In [ ]:
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)
backbone = lib_backbone.Backbone(config=app_config)
backbone.initStorer()
base_dir = app_config.env.dir_base
cfg = app_config.hfss

PARAM_NAMES = list(cfg.param_names)
ROUND_DECIMALS = app_config.runtime.round_decimals

# ===== User settings (x_center may be changed directly) =====
x_center = np.asarray([
    6.059467,  # h1
    1.025253,  # h2
    2.030633,  # h3
    2.794909,  # h4
    9.150354,  # h5
    1.040319,  # s1
    1.138784,  # s2
    0.630056,  # s3
    0.970955,  # s4
    1.996716,  # s5
    2.000000,  # a
    3.566009,  # b
    4.221822,  # k
], dtype=float)
sigma_vec = np.asarray([
    0.1,  # h1
    0.1,  # h2
    0.1,  # h3
    0.1,  # h4
    0.1,  # h5
    0.02,  # s1
    0.02,  # s2
    0.02,  # s3
    0.02,  # s4
    0.02,  # s5
    0.1,  # a
    0.1,  # b
    0.1,  # k
], dtype=float)
PARAM_UNITS = {name: "mm" for name in PARAM_NAMES}
perturbation_enabled = {name: True for name in PARAM_NAMES}
STEP_SCALES = [0.25, 0.5, 1.0]
REFERENCE_STEP_SCALE = 0.5
custom_step = None  # dict keyed by every PARAM_NAME, or a shape-(d,) array
repeats = 1
objective_col = "S11"
OBJECTIVE_UNIT = "dB"
OUTPUT_DIR = backbone._get_dir_run()

print(f"d={len(PARAM_NAMES)}, active={sum(perturbation_enabled.values())}")


In [ ]:
def _enabled_array(param_names, enabled):
    if isinstance(enabled, dict):
        if set(enabled) != set(param_names):
            missing = set(param_names) - set(enabled)
            extra = set(enabled) - set(param_names)
            raise ValueError(f"perturbation_enabled keys must match PARAM_NAMES; missing={missing}, extra={extra}")
        return np.asarray([bool(enabled[name]) for name in param_names])

    output = np.asarray(enabled, dtype=bool).reshape(-1)
    if output.size != len(param_names):
        raise ValueError("perturbation_enabled length must equal len(PARAM_NAMES).")
    return output


def _step_array(param_names, sigma, step_scale, custom):
    if custom is None:
        return float(step_scale) * sigma
    if isinstance(custom, dict):
        if set(custom) != set(param_names):
            raise ValueError("custom_step keys must match PARAM_NAMES exactly.")
        return np.asarray([custom[name] for name in param_names], dtype=float)

    output = np.asarray(custom, dtype=float).reshape(-1)
    if output.size != len(param_names):
        raise ValueError("custom_step length must equal len(PARAM_NAMES).")
    return output


def validate_central_difference_inputs(param_names, center, sigma, enabled, steps, round_decimals):
    center = np.asarray(center, dtype=float).reshape(-1)
    sigma = np.asarray(sigma, dtype=float).reshape(-1)
    steps = np.asarray(steps, dtype=float).reshape(-1)
    dimensions = center.size

    if len(param_names) != dimensions:
        raise ValueError("len(PARAM_NAMES) must equal len(x_center).")
    if sigma.size != dimensions:
        raise ValueError("len(sigma_vec) must equal len(x_center).")
    active = _enabled_array(param_names, enabled)
    if steps.size != dimensions:
        raise ValueError("step_vec length must equal len(x_center).")
    if np.any(sigma[active] <= 0):
        raise ValueError("Every active parameter sigma must be positive.")
    if np.any(steps[active] <= 0):
        raise ValueError("Every active parameter step must be positive.")
    if not np.all(np.isfinite(center)) or not np.all(np.isfinite(sigma)) or not np.all(np.isfinite(steps)):
        raise ValueError("center, sigma, and steps must be finite.")
    if int(round_decimals) != round_decimals or round_decimals < 0:
        raise ValueError("round_decimals must be a non-negative integer.")

    rounded_center = np.round(center, int(round_decimals))
    for parameter_index in np.flatnonzero(active):
        minus = rounded_center.copy()
        plus = rounded_center.copy()
        minus[parameter_index] -= steps[parameter_index]
        plus[parameter_index] += steps[parameter_index]
        minus = np.round(minus, int(round_decimals))
        plus = np.round(plus, int(round_decimals))
        minus_step = rounded_center[parameter_index] - minus[parameter_index]
        plus_step = plus[parameter_index] - rounded_center[parameter_index]
        if minus_step <= 0 or plus_step <= 0 or not np.isclose(minus_step, plus_step, rtol=0.0, atol=10.0 ** (-int(round_decimals)) / 2.0):
            name = param_names[parameter_index]
            raise ValueError(
                f"{name}: the configured step cannot form distinct symmetric perturbations "
                f"at {int(round_decimals)} decimal places; increase sigma/custom_step or precision."
            )

    return center, sigma, active, steps


In [ ]:
def build_central_difference_design(param_names, center, sigma, enabled,
                                    step_scale=0.5, custom_step=None, repeats=1,
                                    round_decimals=6, include_center=False):
    """Build an arbitrary-dimensional, central-difference-only design."""
    if int(repeats) != repeats or repeats < 1:
        raise ValueError("repeats must be a positive integer.")

    sigma = np.asarray(sigma, dtype=float).reshape(-1)
    requested = _step_array(param_names, sigma, step_scale, custom_step)
    center, sigma, active, requested = validate_central_difference_inputs(
        param_names, center, sigma, enabled, requested, round_decimals
    )
    rounded_center = np.round(center, round_decimals)
    logical = []

    if include_center:
        logical.append(dict(
            parameter_index=-1,
            parameter_name="__center__",
            side="center",
            requested_step=0.0,
            actual_step=0.0,
            is_center=True,
            step_scale=float(step_scale),
            _x=rounded_center,
        ))

    for parameter_index, parameter_name in enumerate(param_names):
        if not active[parameter_index]:
            continue

        minus = rounded_center.copy()
        plus = rounded_center.copy()
        minus[parameter_index] -= requested[parameter_index]
        plus[parameter_index] += requested[parameter_index]
        minus = np.round(minus, round_decimals)
        plus = np.round(plus, round_decimals)
        actual_step = plus[parameter_index] - rounded_center[parameter_index]

        for side, point in (("minus", minus), ("plus", plus)):
            logical.append(dict(
                parameter_index=parameter_index,
                parameter_name=parameter_name,
                side=side,
                requested_step=requested[parameter_index],
                actual_step=actual_step,
                is_center=False,
                step_scale=float(step_scale),
                _x=point,
            ))

    physical_points = [tuple(row["_x"]) for row in logical]
    if len(physical_points) != len(set(physical_points)):
        raise ValueError("Duplicate physical points were generated before repeats.")

    rows = []
    for item in logical:
        point = item.pop("_x")
        for _ in range(int(repeats)):
            row = dict(item)
            row.update({name: float(point[index]) for index, name in enumerate(param_names)})
            rows.append(row)
    return pd.DataFrame(rows)


def combine_step_designs(designs, share_center=True):
    """Combine scale designs and retain one physical center per repeat."""
    if not designs:
        raise ValueError("At least one step-scale design is required.")

    combined = pd.concat(designs, ignore_index=True)
    center = combined[combined["is_center"]].copy()
    noncenter = combined[~combined["is_center"]].copy()
    if share_center and len(center):
        metadata_columns = {
            "parameter_index", "parameter_name", "side", "requested_step",
            "actual_step", "is_center", "step_scale",
        }
        parameter_columns = [column for column in combined.columns if column not in metadata_columns]
        if len(center[parameter_columns].drop_duplicates()) != 1:
            raise ValueError("Step-scale designs do not share the same physical center.")
        scale_count = center["step_scale"].nunique()
        if scale_count < 1 or len(center) % scale_count:
            raise ValueError("Every step scale must contain the same number of center repeats.")
        repeat_count = len(center) // scale_count
        center = center.iloc[:repeat_count].copy()
        center["step_scale"] = np.nan

    output = pd.concat([center, noncenter], ignore_index=True)
    return output.sort_values(
        ["is_center", "parameter_index", "step_scale", "side"],
        ascending=[False, True, True, True],
        na_position="first",
    ).reset_index(drop=True)


## Perturbation points

The next code block creates and displays every HFSS input point before any simulation starts. Each active parameter has exactly one `minus` and one `plus` point at every scale. The shared `center` rows provide $f(x)$ for the forward/backward slope asymmetry diagnostic; they are not used in the central-gradient numerator.

The configured `_config.toml` optimization boundaries are deliberately not applied. If rounding makes either perturbation indistinguishable from the center or makes the two step lengths asymmetric, input validation raises an error here, before HFSS starts.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
designs = []
for scale in STEP_SCALES:
    designs.append(build_central_difference_design(
        PARAM_NAMES,
        x_center,
        sigma_vec,
        perturbation_enabled,
        scale,
        custom_step,
        repeats,
        ROUND_DECIMALS,
        include_center=True,
    ))

design_df = combine_step_designs(designs, share_center=True)
design_df.to_csv(OUTPUT_DIR / "central_difference_input.csv", index=False)
print("Predicted HFSS evaluations:", len(design_df))
display(design_df)


In [ ]:
def getResult(temp_hfss_path):
    try:
        df_temp = pd.read_csv(temp_hfss_path)
        s11_value = df_temp.iloc[-1, -1]

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return True, float(s11_value)

    except Exception as error:
        print(f"[Error][getResult] Failed to process result: {error}")
        return False, np.nan


def evaluate_central_difference_design(design, param_names, backbone, config_raw, app_config,
                                       output_dir, objective_col="S11"):
    """Evaluate the complete design once; checkpoint and resume behavior is intentionally absent."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _, model_paths_str = backbone._get_path_models()
    temp_file = str(output_dir / Path(config_raw["io"]["filename_temp"]))
    run_config = {
        "n_simulation": int(len(design)),
        "n_repeats": 1,
        "WATCH_DIR": str(output_dir),
        "INPUT_FILE": str(output_dir / Path(config_raw["io"]["filename_input"])),
        "MODEL_FILE": model_paths_str,
        "RESULTS_FILE": str(output_dir / Path(config_raw["io"]["filename_output"])),
        "TEMP_FILE": temp_file,
        "DONE_FLAG_FILE": str(output_dir / "hfss.done"),
    }
    done_flag_path = Path(run_config["DONE_FLAG_FILE"])
    done_flag_path.unlink(missing_ok=True)

    config_json = app_config.env.dir_base / "_config_HFSS.json"
    with open(config_json, "w") as file:
        json.dump(run_config, file, indent=4)

    records = []
    try:
        for _, metadata in design.iterrows():
            params = metadata[param_names].to_numpy(dtype=float)
            sim_id = backbone._getSimulationID()
            backbone.call_subroutine(
                run_config,
                sim_id,
                param_names,
                params,
                value_fmt=f"{{:.{app_config.runtime.round_decimals}f}}",
            )
            success, s11_value = getResult(temp_file)
            if not success:
                raise RuntimeError(f"HFSS result processing failed for parameters {params.tolist()}.")
            record = metadata.to_dict()
            record[objective_col] = s11_value
            records.append(record)
    finally:
        done_flag_path.touch()
        config_json.unlink(missing_ok=True)

    results = pd.DataFrame(records)
    results.to_csv(output_dir / "central_difference_results.csv", index=False)
    return results


results_df = evaluate_central_difference_design(
    design_df, PARAM_NAMES, backbone, _config, app_config, OUTPUT_DIR, objective_col
)


## Gradient diagnostics and reference scale

For every parameter and step scale:

- `gradient` is the centered slope $(f(x+h)-f(x-h))/(2h)$.
- `forward_slope` is $(f(x+h)-f(x))/h$.
- `backward_slope` is $(f(x)-f(x-h))/h$.
- `asymmetry_abs` is the absolute difference between the forward and backward slopes. A value near zero means that the two sides have similar local slopes; a larger value indicates curvature, simulation noise, or a step that spans a non-local region. For example, forward/backward slopes of `-2.0` and `-2.1 dB/mm` give `asymmetry_abs=0.1 dB/mm`.
- `asymmetry_ratio` divides `asymmetry_abs` by the absolute centered gradient. It is a scale-free diagnostic. For example, `asymmetry_abs=0.1` and `gradient=-2.0` give `asymmetry_ratio=0.05`. When the centered gradient is almost zero, this ratio can be large even for a small absolute asymmetry, so both columns should be interpreted together.

No warning threshold and no second derivative are calculated. The three scale-specific gradients remain available in `step_gradients` for direct inspection.

`REFERENCE_STEP_SCALE=0.5` selects the scale used for the saved sensitivity table and normalized sigma-weighted susceptibility plot. If the exact configured scale is unavailable, the numerically closest scale is selected. The smaller `0.25` scale is more sensitive to rounding and HFSS noise; the larger `1.0` scale is more likely to include non-local behavior. The `0.5` scale is therefore the default reporting compromise, not an automatic claim that it is the most accurate scale.

In [ ]:
def aggregate_repeated_results(results, objective_col="S11"):
    if objective_col not in results:
        raise ValueError(f"HFSS results lack objective_col={objective_col!r}.")

    values = pd.to_numeric(results[objective_col], errors="coerce")
    if not np.all(np.isfinite(values)):
        raise ValueError("All objective values must be finite.")

    aggregated = results.copy()
    aggregated[objective_col] = values
    keys = [
        "parameter_index",
        "parameter_name",
        "side",
        "requested_step",
        "actual_step",
        "is_center",
        "step_scale",
    ]
    return aggregated.groupby(keys, dropna=False, sort=False)[objective_col].agg(
        value_mean="mean",
        value_std="std",
        n_evaluations="size",
    ).reset_index()


def compute_central_difference_gradient(aggregated, param_names, sigma, enabled,
                                        objective_unit="dB", param_units=None, epsilon=1e-12):
    active = _enabled_array(param_names, enabled)
    sigma = np.asarray(sigma, dtype=float)
    param_units = param_units or {}
    center_value = float(aggregated.loc[aggregated["side"].eq("center"), "value_mean"].iloc[0])
    rows = []

    for parameter_index, parameter_name in enumerate(param_names):
        parameter_unit = param_units.get(parameter_name, "unknown")
        base = dict(
            parameter_index=parameter_index,
            parameter_name=parameter_name,
            parameter_unit=parameter_unit,
            objective_unit=objective_unit,
            gradient_unit=(f"{objective_unit} / {parameter_unit}" if parameter_unit != "unknown" else "unknown"),
            sigma=sigma[parameter_index],
        )
        if not active[parameter_index]:
            rows.append(dict(
                base,
                gradient=0.0,
                requested_step=np.nan,
                actual_step=np.nan,
                plus_mean=np.nan,
                minus_mean=np.nan,
                center_value=center_value,
                plus_std=np.nan,
                minus_std=np.nan,
                gradient_standard_error=np.nan,
                forward_slope=np.nan,
                backward_slope=np.nan,
                asymmetry_abs=np.nan,
                asymmetry_ratio=np.nan,
                n_plus_evaluations=0,
                n_minus_evaluations=0,
                status="fixed",
            ))
            continue

        parameter_rows = aggregated[aggregated["parameter_index"].eq(parameter_index)]
        plus = parameter_rows[parameter_rows["side"].eq("plus")].iloc[0]
        minus = parameter_rows[parameter_rows["side"].eq("minus")].iloc[0]
        step = float(plus["actual_step"])
        plus_mean = float(plus["value_mean"])
        minus_mean = float(minus["value_mean"])
        plus_std = float(plus["value_std"])
        minus_std = float(minus["value_std"])
        n_plus = int(plus["n_evaluations"])
        n_minus = int(minus["n_evaluations"])

        gradient = (plus_mean - minus_mean) / (2.0 * step)
        gradient_standard_error = np.sqrt(
            plus_std ** 2 / n_plus + minus_std ** 2 / n_minus
        ) / (2.0 * step)
        forward_slope = (plus_mean - center_value) / step
        backward_slope = (center_value - minus_mean) / step
        asymmetry_abs = abs(forward_slope - backward_slope)
        asymmetry_ratio = asymmetry_abs / max(abs(gradient), epsilon)

        rows.append(dict(
            base,
            gradient=gradient,
            requested_step=float(plus["requested_step"]),
            actual_step=step,
            plus_mean=plus_mean,
            minus_mean=minus_mean,
            center_value=center_value,
            plus_std=plus_std,
            minus_std=minus_std,
            gradient_standard_error=gradient_standard_error,
            forward_slope=forward_slope,
            backward_slope=backward_slope,
            asymmetry_abs=asymmetry_abs,
            asymmetry_ratio=asymmetry_ratio,
            n_plus_evaluations=n_plus,
            n_minus_evaluations=n_minus,
            status="success",
        ))

    return pd.DataFrame(rows)


def compute_sigma_weighted_susceptibility(table):
    output = table.copy()
    gradients = output["gradient"].fillna(0).to_numpy(dtype=float)
    sigma = output["sigma"].to_numpy(dtype=float)
    output["S_raw"] = gradients * gradients
    output["C_sigma"] = (sigma * gradients) ** 2
    output["S_raw_normalized"] = output["S_raw"] / output["S_raw"].sum() if output["S_raw"].sum() > 0 else 0.0
    output["C_sigma_normalized"] = output["C_sigma"] / output["C_sigma"].sum() if output["C_sigma"].sum() > 0 else 0.0
    return output


def plot_central_difference_results(table):
    figure, axis = plt.subplots(figsize=(10, 5))
    axis.bar(table["parameter_name"], table["C_sigma_normalized"])
    axis.set_title("Normalized sigma-weighted susceptibility")
    axis.set_ylabel("C_sigma_normalized")
    axis.tick_params(axis="x", rotation=60)
    figure.tight_layout()
    plt.show()


In [ ]:
# Post-process either the current run or a separately supplied complete result CSV.
results_path = OUTPUT_DIR / "central_difference_results.csv"
if results_path.exists():
    results_df = pd.read_csv(results_path)
    aggregated = aggregate_repeated_results(results_df, objective_col)
    shared_center = aggregated[aggregated["side"].eq("center")]
    scales = sorted(aggregated.loc[~aggregated["side"].eq("center"), "step_scale"].dropna().unique())
    tables = {}

    for scale in scales:
        scale_aggregated = pd.concat(
            [shared_center, aggregated[aggregated["step_scale"].eq(scale)]],
            ignore_index=True,
        )
        tables[scale] = compute_central_difference_gradient(
            scale_aggregated,
            PARAM_NAMES,
            sigma_vec,
            perturbation_enabled,
            OBJECTIVE_UNIT,
            PARAM_UNITS,
        )

    step_gradients = pd.concat(
        {scale: table.set_index("parameter_name")["gradient"] for scale, table in tables.items()},
        axis=1,
    )
    reference = min(tables, key=lambda scale: abs(float(scale) - REFERENCE_STEP_SCALE))
    gradient_df = compute_sigma_weighted_susceptibility(tables[reference])
    gradient_df.to_csv(OUTPUT_DIR / "central_difference_gradient.csv", index=False)

    summary = {
        "parameter_names": PARAM_NAMES,
        "x_center": x_center.tolist(),
        "sigma_vec": sigma_vec.tolist(),
        "step_scales": STEP_SCALES,
        "reference_step_scale": float(reference),
        "round_decimals": ROUND_DECIMALS,
        "repeats": repeats,
        "predicted_evaluations": len(design_df),
        "actual_rows_saved": len(results_df),
        "objective_col": objective_col,
    }
    with open(OUTPUT_DIR / "central_difference_summary.json", "w") as file:
        json.dump(summary, file, indent=2)

    display(gradient_df)
    display(step_gradients)
    for row in gradient_df.itertuples():
        print(f"∂{objective_col} / ∂{row.parameter_name} = {row.gradient:.8g} {row.gradient_unit}")
    plot_central_difference_results(gradient_df)
